# Unitary Block Encoding

Adapted from Qualtran v0.7.0, commit bcb17bb25360ed68ce3bc084645db3e929799389.
Copyright Google LLC. SPDX-License-Identifier: Apache-2.0.

In [ ]:
from dataclasses import replace

import torch
from IPython.display import SVG, display

from examples.algorithms.field_addition_catalog import field_signature_svg
from flagquantum import Circuit
from flagquantum.compiler._component_trace.resource_model import ComponentCost, GateCost
from flagquantum.core.ir import Instruction
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_modeled_call_graph_dot, render_call_graph_svg
from flagquantum.simulation.unitary import get_unitary

## Wrap a unitary child

For a unitary $U$, the default block encoding is $B[U] = U$ with scale
one, zero ancilla bits, and zero approximation error. The typed child
signature admits through ports only. Pack those ports in declaration
order into the system word. Each register word uses big-endian bit order.

Override the scale, ancilla width, resource width and precision when
building a larger block encoding. The declared auxiliary words pass
through the child untouched. Zero-width auxiliary words are omitted.
The caller supplies any temporary work and result slots required by
the child decomposition.

In [ ]:
from flagquantum.algorithms.primitives.unitary_block_encoding import (
    UnitaryBlockEncoding,
)

### Example Instances

In [ ]:
def emit_t(ports, work, slots, adjoint):
    return (Instruction("tdg" if adjoint else "t", ports["q"]),)


plain = UnitaryBlockEncoding.from_signature(Signature.build(q=1), emit_t)
assert (plain.alpha, plain.epsilon) == (1, 0)

In [ ]:
overrides = replace(plain, alpha=0.5, ancilla_bitsize=2,
                    resource_bitsize=1, epsilon=0.01)
assert (overrides.alpha, overrides.epsilon) == (0.5, 0.01)

#### Graphical Signature

In [ ]:
display(SVG(field_signature_svg([
    ("Unitary T", plain.signature, {}),
    ("Unitary T with overrides", overrides.signature, {}),
])))

### Call Graph

In [ ]:
model = plain.resource_model(ComponentCost("T", 1, GateCost(t=1)))
display(SVG(render_call_graph_svg(export_modeled_call_graph_dot(model, max_depth=1))))
print([(call.component.name, call.repetitions) for call in model.calls])

### Append and verify both wrappers

The full matrices verify the relative T phase for all input basis
states. The overridden case preserves the phase on the system and
acts as identity across all eight auxiliary states.

In [ ]:
t_matrix = torch.diag(torch.tensor(
    [1, complex(2**-0.5, 2**-0.5)], dtype=torch.complex128))
for encoding, ancilla, resource in ((plain, (), ()), (overrides, (1, 2), (3,))):
    auxiliary = len(ancilla) + len(resource)
    ports = dict(system=(0,), ancilla=ancilla, resource=resource)
    circuit = Circuit(1 + auxiliary, dtype=torch.complex128)
    circuit.append(encoding.make_on(**ports))
    expected = (torch.kron(t_matrix, torch.eye(2**auxiliary, dtype=torch.complex128))
                if auxiliary else t_matrix)
    torch.testing.assert_close(get_unitary(circuit), expected)
    circuit.append(encoding.make_on(**ports, adjoint=True))
    torch.testing.assert_close(get_unitary(circuit),
                               torch.eye(2**(1 + auxiliary), dtype=torch.complex128))